# M4S4 - Distributed Machine Learning and AI Workflows
## From data ingestion to training, serving, monitoring and recovery

A scalable machine-learning system is more than a training loop.

In this notebook we will build and measure small pieces of a real workflow:

**data -> ingest -> preprocess -> train -> evaluate -> select -> serve -> monitor**

We will also see:

- batching, sharding and caching;
- parameter-server communication;
- real PyTorch DistributedDataParallel;
- fan-out and fan-in;
- local model serving;
- monitoring metadata;
- checkpoint and restart;
- a small workflow orchestrator;
- the resources visible on the HPC cluster.

**Predict -> run -> observe -> explain**


### How to use the notebook

Run the cells in order.

Most examples use only Python and NumPy and are safe on a CPU node. The PyTorch DDP example uses the cluster module system when it is available.

The examples deliberately stay small so that we can focus on the distributed-systems pattern rather than model accuracy.


## 0 - Check the environment

Before building a distributed workflow, inspect the machine and the cluster interfaces that are visible from this Jupyter session.


In [ ]:
import os
import sys
import json
import time
import math
import shutil
import socket
import hashlib
import subprocess
import threading
import urllib.request
import urllib.error
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np

OUT = Path("m4s4_outputs")
OUT.mkdir(exist_ok=True)

def sh(cmd):
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    return (p.stdout + p.stderr).strip()

print("host:", sh("hostname"))
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("CPU cores visible:", os.cpu_count())
print("Slurm visible:", bool(shutil.which("sinfo")))
print("nvidia-smi visible:", bool(shutil.which("nvidia-smi")))

if shutil.which("nvidia-smi"):
    print("\nVisible GPU information:")
    print(sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | head -8"))

def run_cluster_command(module_candidates, command):
    bootstrap = r"""
if ! type module >/dev/null 2>&1; then
  for f in /etc/profile.d/lmod.sh /etc/profile.d/modules.sh /usr/share/lmod/lmod/init/bash; do
    if [ -r "$f" ]; then source "$f"; break; fi
  done
fi
"""
    for module_name in module_candidates:
        shell = bootstrap + f"""
module purge >/dev/null 2>&1 || true
if ! module load {module_name} >/dev/null 2>&1; then
  exit 97
fi
echo "module: {module_name}"
echo "python: $(command -v python)"
{command}
"""
        p = subprocess.run(["bash", "-lc", shell], text=True, capture_output=True)
        output = (p.stdout + p.stderr).strip()
        if p.returncode == 0:
            print(output)
            return {"ok": True, "module": module_name, "output": output}
    print("No matching framework module could be executed from this Jupyter session.")
    return {"ok": False}

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def accuracy_from_logits(logits, y):
    return float((logits.argmax(axis=1) == y).mean())


## 1 - Build a tiny end-to-end ML dataset

We will use one synthetic classification dataset throughout several examples.

The important point is not the model. The important point is that the same dataset becomes an **artifact** that moves through several workflow stages.


In [ ]:
rng = np.random.default_rng(19)

N, D, C = 7000, 32, 3
X = rng.normal(size=(N, D)).astype(np.float32)
teacher_W = rng.normal(size=(D, C)).astype(np.float32)
teacher_b = rng.normal(size=(C,)).astype(np.float32)
noise = 0.35 * rng.normal(size=(N, C)).astype(np.float32)

y = np.argmax(X @ teacher_W + teacher_b + noise, axis=1).astype(np.int64)

train_end = 5000
val_end = 6000

dataset_file = OUT / "workflow_dataset.npz"
np.savez_compressed(
    dataset_file,
    X=X,
    y=y,
    train_end=train_end,
    val_end=val_end,
)

sha256 = hashlib.sha256(dataset_file.read_bytes()).hexdigest()

print("dataset:", dataset_file)
print("shape:", X.shape)
print("classes:", np.bincount(y))
print("train / validation / test:", train_end, val_end-train_end, N-val_end)
print("compressed size:", f"{dataset_file.stat().st_size/1e6:.2f} MB")
print("SHA256:", sha256[:20] + "...")


**Question:** Why is the dataset hash useful?

A distributed workflow may run many jobs at different times. A content hash helps us verify that two runs used the same input artifact.


## 2 - [DEMO] Batching changes the unit of work

A batch groups several samples into one compute step.

We will process the same matrix workload with different batch sizes and measure:

- number of batches;
- elapsed time;
- examples per second.

This is a real computation. The best batch size can depend on the hardware and the operation.


In [ ]:
rng = np.random.default_rng(2)
Xb = rng.normal(size=(24000, 128)).astype(np.float32)
Wb = rng.normal(size=(128, 128)).astype(np.float32)

def run_batches(batch_size):
    t0 = time.perf_counter()
    checksum = 0.0
    n_batches = 0
    for start in range(0, len(Xb), batch_size):
        batch = Xb[start:start+batch_size]
        out = np.tanh(batch @ Wb)
        checksum += float(out[0, 0])
        n_batches += 1
    dt = time.perf_counter() - t0
    return n_batches, dt, len(Xb)/dt, checksum

batch_results = []
for bs in [16, 64, 256, 1024]:
    n_batches, dt, throughput, checksum = run_batches(bs)
    batch_results.append((bs, n_batches, dt, throughput))
    print(
        f"batch={bs:4d} | batches={n_batches:4d} | "
        f"time={dt:.4f}s | throughput={throughput:,.0f} examples/s"
    )


### Read the result

A larger batch reduces the number of loop iterations and often improves hardware utilization, but it also increases the amount of data held for each step.

**Batching is about the unit of processing. It is not the same as sharding a dataset across workers.**


## 3 - [DEMO] Sharding distributes the dataset

Now split the training set into four physical shard files.

Each shard contains a different subset of examples. This is a common pattern when several workers need to read different portions of a large dataset.


In [ ]:
data = np.load(dataset_file)
X_all = data["X"]
y_all = data["y"]

train_ids = np.arange(train_end)
shard_ids = np.array_split(train_ids, 4)

shard_files = []
for rank, ids in enumerate(shard_ids):
    p = OUT / f"train_shard_{rank}.npz"
    np.savez_compressed(p, ids=ids, X=X_all[ids], y=y_all[ids])
    shard_files.append(p)
    print(
        f"shard {rank}: rows={len(ids):4d} | "
        f"id range={ids.min()}..{ids.max()} | "
        f"file={p.name}"
    )

all_ids = np.concatenate([np.load(p)["ids"] for p in shard_files])

print("\ncoverage correct:", np.array_equal(np.sort(all_ids), train_ids))
print("duplicates:", len(all_ids) - len(np.unique(all_ids)))
print("total examples:", len(all_ids))


**Question:** If four training workers each read one shard, are they automatically training one shared model?

No. Sharding only distributes data. The training processes still need a communication strategy if they are collaborating on one model.


## 4 - [REAL I/O DEMO] Cache expensive preprocessing

Suppose every training epoch repeats the same deterministic preprocessing.

We will compare:

1. loading compressed raw data and recomputing the transform every epoch;
2. preprocessing once, storing the result, then reusing the cached artifact.

The exact timing will vary, but the work and file I/O are real.


In [ ]:
rng = np.random.default_rng(4)
raw_X = rng.normal(size=(20000, 96)).astype(np.float32)
projection = rng.normal(size=(96, 96)).astype(np.float32)

raw_file = OUT / "raw_features.npz"
cache_file = OUT / "cached_features.npy"

np.savez_compressed(raw_file, X=raw_X)

def preprocess(x):
    return np.tanh(x @ projection).astype(np.float32)

recompute_times = []
for epoch in range(3):
    t0 = time.perf_counter()
    raw = np.load(raw_file)["X"]
    transformed = preprocess(raw)
    checksum = float(transformed[0, 0])
    recompute_times.append(time.perf_counter() - t0)

t0 = time.perf_counter()
raw = np.load(raw_file)["X"]
cached = preprocess(raw)
np.save(cache_file, cached)
cache_build_time = time.perf_counter() - t0

cached_times = []
for epoch in range(3):
    t0 = time.perf_counter()
    transformed = np.load(cache_file, mmap_mode="r")
    checksum = float(transformed[0, 0])
    cached_times.append(time.perf_counter() - t0)

print("recompute each epoch:", [f"{x:.4f}s" for x in recompute_times])
print("one-time cache build:", f"{cache_build_time:.4f}s")
print("reuse cached artifact:", [f"{x:.4f}s" for x in cached_times])
print(
    "3-epoch repeated work:",
    f"recompute={sum(recompute_times):.4f}s | cached reads={sum(cached_times):.4f}s"
)


### What changed?

Caching did not make the transformation itself faster.

It changed the workflow:

**compute the reusable result once -> store it -> reuse it**

This trades storage space for less repeated computation.


## 5 - [DEMO] Prefetch can overlap file loading with computation

A pipeline can often load the next shard while the current shard is being processed.

We will compare:

**sequential:** load 1 -> compute 1 -> load 2 -> compute 2

with:

**prefetch:** load 1 -> compute 1 while loading 2 -> compute 2 while loading 3

This uses real compressed files and a background loading thread.


In [ ]:
prefetch_dir = OUT / "prefetch_shards"
prefetch_dir.mkdir(exist_ok=True)

rng = np.random.default_rng(5)
P = rng.normal(size=(96, 64)).astype(np.float32)

prefetch_files = []
for i in range(8):
    arr = rng.normal(size=(3500, 96)).astype(np.float32)
    p = prefetch_dir / f"part_{i}.npz"
    np.savez_compressed(p, X=arr)
    prefetch_files.append(p)

def load_part(path):
    return np.load(path)["X"]

def compute_part(x):
    y = np.tanh(x @ P)
    return float(np.linalg.norm(y[:200]))

def run_sequential(paths):
    t0 = time.perf_counter()
    checks = []
    for p in paths:
        x = load_part(p)
        checks.append(compute_part(x))
    return time.perf_counter()-t0, checks

def run_prefetched(paths):
    t0 = time.perf_counter()
    checks = []
    with ThreadPoolExecutor(max_workers=1) as ex:
        future = ex.submit(load_part, paths[0])
        for i, p in enumerate(paths):
            x = future.result()
            if i + 1 < len(paths):
                future = ex.submit(load_part, paths[i+1])
            checks.append(compute_part(x))
    return time.perf_counter()-t0, checks

seq_time, seq_checks = run_sequential(prefetch_files)
pre_time, pre_checks = run_prefetched(prefetch_files)

print(f"sequential: {seq_time:.4f}s")
print(f"prefetched: {pre_time:.4f}s")
print(f"speed ratio sequential/prefetch: {seq_time/pre_time:.2f}x")
print("same numerical work:", np.allclose(seq_checks, pre_checks))


Prefetching improves performance only when useful loading work can overlap with useful computation.

If the storage, CPU or computation behaves differently, the measured benefit can be small or even disappear.

**Measure the actual pipeline.**


## 6 - [REAL MULTIPROCESS DEMO] A tiny parameter server

A parameter-server architecture has a central owner of model state.

In this example:

- four worker processes receive the current weights;
- each worker computes a gradient on its own shard;
- workers send gradients to the server process;
- the server averages the gradients and updates the weights;
- the updated weights are sent out for the next step.

The communication uses real operating-system processes and queues.


In [ ]:
ps_script = OUT / "parameter_server_demo.py"

ps_script.write_text(r'''
import multiprocessing as mp
import numpy as np

def worker(rank, X, y, weight_q, grad_q):
    while True:
        w = weight_q.get()
        if w is None:
            break
        pred = X @ w
        err = pred - y
        grad = (X.T @ err) / len(X)
        loss = float(np.mean(err**2))
        grad_q.put((rank, grad, loss))

if __name__ == "__main__":
    ctx = mp.get_context("fork")
    rng = np.random.default_rng(6)

    n, d, workers = 4000, 24, 4
    X = rng.normal(size=(n, d))
    true_w = rng.normal(size=d)
    y = X @ true_w + 0.10 * rng.normal(size=n)

    ids = np.array_split(np.arange(n), workers)
    weight_qs = [ctx.Queue() for _ in range(workers)]
    grad_q = ctx.Queue()

    procs = []
    for rank in range(workers):
        p = ctx.Process(
            target=worker,
            args=(rank, X[ids[rank]], y[ids[rank]], weight_qs[rank], grad_q)
        )
        p.start()
        procs.append(p)

    w = np.zeros(d)
    lr = 0.20

    print("parameter server process: owns the shared weights")
    print("workers:", workers)

    for step in range(1, 7):
        for q in weight_qs:
            q.put(w.copy())

        received = [grad_q.get() for _ in range(workers)]
        received.sort(key=lambda x: x[0])

        grads = [item[1] for item in received]
        losses = [item[2] for item in received]
        w -= lr * np.mean(grads, axis=0)

        print(
            f"step {step}: received {len(grads)} gradients | "
            f"mean worker loss={np.mean(losses):.4f}"
        )

    for q in weight_qs:
        q.put(None)
    for p in procs:
        p.join()

    print("final weight error:", float(np.linalg.norm(w-true_w)))
''')

p = subprocess.run(
    [sys.executable, str(ps_script)],
    text=True,
    capture_output=True,
    timeout=60,
)
print(p.stdout)
if p.stderr.strip():
    print(p.stderr)
print("return code:", p.returncode)


### What should you notice?

The workers do not update the shared model directly.

They communicate with a **central state owner**.

That central point simplifies coordination, but it can also become a communication or availability bottleneck as the system grows.


## 7 - [REAL DISTRIBUTED ML] PyTorch DDP and All-Reduce

Now use a different communication pattern.

PyTorch DistributedDataParallel normally gives each process a model replica. During backpropagation, the replicas synchronize gradients through collective communication.

This example launches **two real PyTorch processes**.

If two GPUs are available to the job, it uses NCCL. Otherwise it uses Gloo on CPUs, so the distributed mechanism can still be executed and observed.


In [ ]:
ddp_script = OUT / "m4s4_ddp_demo.py"

ddp_script.write_text(r'''
import os
import time
import torch
import torch.distributed as dist
from torch import nn
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset, DistributedSampler

dist.init_process_group(
    backend="nccl"
    if torch.cuda.is_available() and torch.cuda.device_count() >= dist.get_world_size()
    else "gloo"
)

rank = dist.get_rank()
world = dist.get_world_size()
local_rank = int(os.environ.get("LOCAL_RANK", rank))

use_gpu = dist.get_backend() == "nccl"
device = torch.device(f"cuda:{local_rank}" if use_gpu else "cpu")
if use_gpu:
    torch.cuda.set_device(local_rank)

torch.manual_seed(19)

N, D, C = 6000, 32, 3
X = torch.randn(N, D)
teacher = torch.randn(D, C)
y = (X @ teacher + 0.15 * torch.randn(N, C)).argmax(dim=1)

dataset = TensorDataset(X, y)
sampler = DistributedSampler(dataset, num_replicas=world, rank=rank, shuffle=True)
loader = DataLoader(dataset, batch_size=128, sampler=sampler, num_workers=0)

model = nn.Sequential(
    nn.Linear(D, 64),
    nn.ReLU(),
    nn.Linear(64, C),
).to(device)

ddp = DDP(model, device_ids=[local_rank] if use_gpu else None)
optimizer = torch.optim.SGD(ddp.parameters(), lr=0.15)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(3):
    sampler.set_epoch(epoch)
    total_loss = 0.0
    total_count = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(ddp(xb), yb)
        loss.backward()
        optimizer.step()

        total_loss += float(loss) * xb.shape[0]
        total_count += xb.shape[0]

    pair = torch.tensor([total_loss, total_count], dtype=torch.float64, device=device)
    dist.all_reduce(pair, op=dist.ReduceOp.SUM)

    if rank == 0:
        print(f"epoch {epoch+1}: global loss={pair[0].item()/pair[1].item():.4f}")

checksum = torch.tensor(
    [sum(float(p.detach().float().sum()) for p in ddp.module.parameters())],
    dtype=torch.float64,
    device=device,
)
checks = [torch.zeros_like(checksum) for _ in range(world)]
dist.all_gather(checks, checksum)

payload = torch.ones(250_000, dtype=torch.float32, device=device)
dist.barrier()
if use_gpu:
    torch.cuda.synchronize()

t0 = time.perf_counter()
for _ in range(10):
    x = payload.clone()
    dist.all_reduce(x, op=dist.ReduceOp.SUM)

if use_gpu:
    torch.cuda.synchronize()

allreduce_ms = (time.perf_counter() - t0) * 1000 / 10

with torch.no_grad():
    pred = ddp.module(X.to(device)).argmax(dim=1).cpu()
accuracy = float((pred == y).float().mean())

if rank == 0:
    values = [v.item() for v in checks]
    print("backend:", dist.get_backend())
    print("world size:", world)
    print("device mode:", "GPU" if use_gpu else "CPU")
    print(f"final accuracy: {accuracy:.3f}")
    print(f"replica checksum spread: {max(values)-min(values):.3e}")
    print(f"1 MB All-Reduce average: {allreduce_ms:.3f} ms")

dist.destroy_process_group()
''')

ddp_result = run_cluster_command(
    ["pytorch/2.8.0", "PyTorch/2.1.2-foss-2023a"],
    f'python -m torch.distributed.run --standalone --nproc_per_node=2 "{ddp_script}"'
)


### Evidence of real distributed execution

Look for:

- **world size = 2**: two training processes;
- **backend = gloo or nccl**: a real distributed backend;
- changing global training loss;
- **replica checksum spread near zero**: synchronized model replicas;
- measured time for a real **All-Reduce**.

Compare this architecture with the parameter-server example.

**Question:** Where is the central gradient server in DDP?

There is not one. The participating processes use collective communication.


## 8 - [REAL WORKFLOW] Fan-out: train several independent candidates

Distributed workflows also contain parallelism **between experiments**.

We will launch four independent operating-system processes with different learning rates.

They do not collaborate on one model. Each process trains its own candidate and writes:

- validation accuracy;
- runtime;
- model weights;
- run metadata.

This is workflow parallelism.


In [ ]:
candidate_script = OUT / "train_candidate.py"

candidate_script.write_text(r'''
import argparse
import json
import os
import time
from pathlib import Path
import numpy as np

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

parser = argparse.ArgumentParser()
parser.add_argument("--data", required=True)
parser.add_argument("--lr", type=float, required=True)
parser.add_argument("--name", required=True)
parser.add_argument("--out", required=True)
args = parser.parse_args()

data = np.load(args.data)
X = data["X"].astype(np.float64)
y = data["y"].astype(np.int64)
train_end = int(data["train_end"])
val_end = int(data["val_end"])

Xtr, ytr = X[:train_end], y[:train_end]
Xv, yv = X[train_end:val_end], y[train_end:val_end]

classes = int(y.max()) + 1
W = np.zeros((X.shape[1], classes), dtype=np.float64)
b = np.zeros(classes, dtype=np.float64)

Y = np.eye(classes)[ytr]

t0 = time.perf_counter()
for epoch in range(80):
    probs = softmax(Xtr @ W + b)
    err = (probs - Y) / len(Xtr)
    W -= args.lr * (Xtr.T @ err)
    b -= args.lr * err.sum(axis=0)

elapsed = time.perf_counter() - t0
val_pred = np.argmax(Xv @ W + b, axis=1)
val_acc = float((val_pred == yv).mean())

out = Path(args.out)
out.mkdir(parents=True, exist_ok=True)
np.savez(out / f"{args.name}_model.npz", W=W, b=b)

metrics = {
    "name": args.name,
    "learning_rate": args.lr,
    "validation_accuracy": val_acc,
    "runtime_s": elapsed,
    "pid": os.getpid(),
}
(out / f"{args.name}_metrics.json").write_text(json.dumps(metrics, indent=2))

print(json.dumps(metrics))
''')

candidate_dir = OUT / "candidates"
candidate_dir.mkdir(exist_ok=True)

learning_rates = [0.005, 0.02, 0.08, 0.20]
procs = []
t0 = time.perf_counter()

for i, lr in enumerate(learning_rates):
    name = f"candidate_{i}"
    env = os.environ.copy()
    env["OMP_NUM_THREADS"] = "1"
    env["OPENBLAS_NUM_THREADS"] = "1"
    env["MKL_NUM_THREADS"] = "1"

    p = subprocess.Popen(
        [
            sys.executable,
            str(candidate_script),
            "--data", str(dataset_file),
            "--lr", str(lr),
            "--name", name,
            "--out", str(candidate_dir),
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        env=env,
    )
    procs.append((name, p))

fanout_results = []
for name, p in procs:
    out, err = p.communicate(timeout=60)
    if p.returncode != 0:
        raise RuntimeError(f"{name} failed:\n{err}")
    result = json.loads(out.strip().splitlines()[-1])
    fanout_results.append(result)
    print(
        f"{name}: pid={result['pid']} | lr={result['learning_rate']} | "
        f"val_acc={result['validation_accuracy']:.3f} | "
        f"runtime={result['runtime_s']:.3f}s"
    )

fanout_wall_s = time.perf_counter() - t0
print("\nfan-out wall time:", f"{fanout_wall_s:.3f}s")
print("sum of candidate runtimes:", f"{sum(r['runtime_s'] for r in fanout_results):.3f}s")


### What proves this was fan-out?

The candidates have different process IDs and independent model artifacts.

The workflow waited for all of them before the next stage.

This is different from DDP:

- **DDP:** several processes cooperate on one model;
- **fan-out:** several jobs train different models independently.


## 9 - [REAL WORKFLOW] Fan-in: evaluate and select one model

Fan-in combines results from parallel branches.

We will read the metrics produced by the four independent training jobs, select the best validation result, copy its model artifact, and evaluate it on data that the candidate jobs did not use.


In [ ]:
metrics = []
for p in sorted(candidate_dir.glob("*_metrics.json")):
    metrics.append(json.loads(p.read_text()))

metrics.sort(key=lambda x: x["validation_accuracy"], reverse=True)

print("candidate ranking:")
for m in metrics:
    print(
        f"{m['name']:12s} | lr={m['learning_rate']:<5} | "
        f"val_acc={m['validation_accuracy']:.3f} | runtime={m['runtime_s']:.3f}s"
    )

best = metrics[0]
best_model_file = candidate_dir / f"{best['name']}_model.npz"
selected_model_file = OUT / "selected_model.npz"
selected_model_file.write_bytes(best_model_file.read_bytes())

model = np.load(selected_model_file)
W_selected = model["W"]
b_selected = model["b"]

data = np.load(dataset_file)
X_all = data["X"].astype(np.float64)
y_all = data["y"].astype(np.int64)

test_logits = X_all[val_end:] @ W_selected + b_selected
test_accuracy = accuracy_from_logits(test_logits, y_all[val_end:])

print("\nselected:", best["name"])
print("selected validation accuracy:", f"{best['validation_accuracy']:.3f}")
print("independent test accuracy:", f"{test_accuracy:.3f}")
print("artifact:", selected_model_file)


**Question:** Why did we use validation accuracy to choose the model and keep the test set separate?

Model selection is part of the workflow. If the test set is repeatedly used to choose models, it stops being an independent final evaluation.


## 10 - [REAL SERVICE] Serve the selected model over HTTP

Training and serving have different performance objectives.

We will start a tiny local HTTP prediction service that loads the selected model artifact.

Then the notebook will send real JSON requests to it and measure request latency.


In [ ]:
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

served = np.load(selected_model_file)
SERVE_W = served["W"]
SERVE_B = served["b"]

class PredictHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        if self.path != "/predict":
            self.send_response(404)
            self.end_headers()
            return

        length = int(self.headers.get("Content-Length", "0"))
        payload = json.loads(self.rfile.read(length).decode("utf-8"))
        x = np.asarray(payload["x"], dtype=np.float64)
        x = np.atleast_2d(x)

        logits = x @ SERVE_W + SERVE_B
        pred = logits.argmax(axis=1).tolist()

        body = json.dumps({"predictions": pred}).encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass

server = ThreadingHTTPServer(("127.0.0.1", 0), PredictHandler)
port = server.server_address[1]
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()

service_url = f"http://127.0.0.1:{port}/predict"
print("service:", service_url)

sample = X_all[val_end].tolist()
request_body = json.dumps({"x": sample}).encode("utf-8")

latencies_ms = []
for _ in range(30):
    req = urllib.request.Request(
        service_url,
        data=request_body,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    t0 = time.perf_counter()
    with urllib.request.urlopen(req, timeout=5) as r:
        response = json.loads(r.read().decode("utf-8"))
    latencies_ms.append((time.perf_counter()-t0)*1000)

print("example prediction:", response)
print("requests:", len(latencies_ms))
print("median latency:", f"{np.median(latencies_ms):.3f} ms")
print("p95 latency:", f"{np.percentile(latencies_ms,95):.3f} ms")


## 10.1 - [DEMO] Serving batches trade latency for throughput

The same endpoint accepts one example or a list of examples.

We will send different request batch sizes and measure:

- request latency;
- examples processed per second.

On a local tiny model, the numbers will be small and noisy. The important idea is that **request latency and throughput are not the same objective**.


In [ ]:
serving_results = []

for batch_size in [1, 8, 32, 128]:
    payload = {"x": X_all[val_end:val_end+batch_size].tolist()}
    body = json.dumps(payload).encode("utf-8")

    request_times = []
    total_examples = 0
    for _ in range(25):
        req = urllib.request.Request(
            service_url,
            data=body,
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        t0 = time.perf_counter()
        with urllib.request.urlopen(req, timeout=5) as r:
            result = json.loads(r.read().decode("utf-8"))
        dt = time.perf_counter()-t0
        request_times.append(dt)
        total_examples += len(result["predictions"])

    total_time = sum(request_times)
    median_ms = 1000*np.median(request_times)
    throughput = total_examples/total_time

    serving_results.append({
        "batch_size": batch_size,
        "median_request_ms": float(median_ms),
        "examples_per_s": float(throughput),
    })

    print(
        f"batch={batch_size:3d} | median request={median_ms:7.3f} ms | "
        f"throughput={throughput:9.0f} examples/s"
    )

server.shutdown()
server_thread.join(timeout=5)
print("\nservice stopped")


### Training metric versus serving metric

For training we often care about:

**time-to-train, examples/s, accelerator utilization**

For serving we often care about:

**latency, requests/s, availability, error rate**

The same model can be successful under one objective and poor under another.


## 11 - [REAL WORKFLOW] Monitoring and metadata

A distributed workflow produces more than model weights.

We also want enough metadata to answer:

- which dataset was used?
- which candidate was selected?
- what configuration produced it?
- how long did training take?
- what did serving latency look like?
- can we reproduce the result?

We will create a small run record from the artifacts we have produced.


In [ ]:
workflow_record = {
    "dataset": {
        "path": str(dataset_file),
        "sha256": sha256,
        "rows": int(N),
        "features": int(D),
    },
    "fan_out": {
        "candidates": metrics,
        "wall_time_s": fanout_wall_s,
    },
    "selection": {
        "candidate": best["name"],
        "learning_rate": best["learning_rate"],
        "validation_accuracy": best["validation_accuracy"],
        "test_accuracy": test_accuracy,
        "artifact": str(selected_model_file),
    },
    "serving": {
        "single_request_median_ms": float(np.median(latencies_ms)),
        "single_request_p95_ms": float(np.percentile(latencies_ms, 95)),
        "batch_results": serving_results,
    },
    "environment": {
        "host": sh("hostname"),
        "python": sys.version.split()[0],
        "numpy": np.__version__,
    },
}

record_file = OUT / "workflow_record.json"
record_file.write_text(json.dumps(workflow_record, indent=2))

print(record_file.read_text()[:1800])
print("\nrecord saved:", record_file)


**Question:** GPU utilization is 95%, but model quality has degraded after deployment. Is the system healthy?

Not necessarily.

Monitoring needs several views:

- **system metrics**: CPU, GPU, memory, storage, network;
- **model metrics**: loss, accuracy or task quality;
- **service metrics**: latency, throughput, errors;
- **data metrics**: availability, distribution and drift.


## 12 - [REAL FAILURE DEMO] Checkpoint, fail and resume

At scale, a long-running training job can fail.

We will intentionally stop a small optimization after step 23.

The job checkpoints every 10 steps, so the restart should recover from step 20 rather than from the beginning.


In [ ]:
checkpoint_file = OUT / "fault_demo_checkpoint.npz"

rng = np.random.default_rng(12)
Xf = rng.normal(size=(3000, 20))
true_wf = rng.normal(size=20)
yf = Xf @ true_wf + 0.05*rng.normal(size=len(Xf))

def one_step(w, lr=0.08):
    err = Xf @ w - yf
    grad = (Xf.T @ err) / len(Xf)
    return w - lr*grad, float(np.mean(err**2))

def save_checkpoint(step, w):
    np.savez(checkpoint_file, step=step, w=w)

w = np.zeros(Xf.shape[1])
failure_step = 23
checkpoint_every = 10

try:
    for step in range(1, 41):
        w, loss = one_step(w)

        if step % checkpoint_every == 0:
            save_checkpoint(step, w)
            print(f"checkpoint saved at step {step}, loss={loss:.5f}")

        if step == failure_step:
            raise RuntimeError("simulated worker failure")

except RuntimeError as e:
    print("\nfailure:", e)

saved = np.load(checkpoint_file)
resume_step = int(saved["step"])
w = saved["w"].copy()

print("recovered checkpoint step:", resume_step)

for step in range(resume_step+1, 41):
    w, loss = one_step(w)

print("resumed final step: 40")
print("resumed final loss:", f"{loss:.6f}")


In [ ]:
w_ref = np.zeros(Xf.shape[1])
for step in range(1, 41):
    w_ref, ref_loss = one_step(w_ref)

difference = float(np.linalg.norm(w - w_ref))

print("uninterrupted final loss:", f"{ref_loss:.6f}")
print("difference between resumed and uninterrupted weights:", f"{difference:.3e}")
print("work lost after failure:", failure_step - resume_step, "steps")


### Checkpointing is a trade-off

Checkpoint very frequently:

- less lost work after failure;
- more I/O overhead.

Checkpoint rarely:

- less I/O;
- more recomputation after failure.

The right interval depends on job duration, failure risk, checkpoint size and storage performance.


## 13 - [REAL WORKFLOW] A tiny dependency-aware orchestrator

A workflow orchestrator decides **what can run next** based on dependencies.

This is different from a cluster scheduler such as Slurm, which decides **where and when a job gets resources**.

We will execute this small DAG:

**ingest -> preprocess -> {train_A, train_B, train_C} -> select -> register**

The three training branches are allowed to run concurrently.


In [ ]:
orch_dir = OUT / "orchestrator_demo"
orch_dir.mkdir(exist_ok=True)

rng = np.random.default_rng(13)
orch_X = rng.normal(size=(2400, 16))
orch_true = rng.normal(size=(16, 2))
orch_y = np.argmax(orch_X @ orch_true + 0.2*rng.normal(size=(2400,2)), axis=1)

def task_ingest():
    p = orch_dir / "raw.npz"
    np.savez(p, X=orch_X, y=orch_y)
    return str(p)

def task_preprocess():
    d = np.load(orch_dir / "raw.npz")
    x = d["X"]
    x = (x - x.mean(axis=0)) / (x.std(axis=0) + 1e-8)
    p = orch_dir / "prepared.npz"
    np.savez(p, X=x, y=d["y"])
    return str(p)

def task_train(name, lr):
    d = np.load(orch_dir / "prepared.npz")
    x, y = d["X"], d["y"]
    train = 1800
    classes = 2
    W = np.zeros((x.shape[1], classes))
    Y = np.eye(classes)[y[:train]]

    for _ in range(35):
        probs = softmax(x[:train] @ W)
        grad = x[:train].T @ ((probs-Y)/train)
        W -= lr*grad

    acc = accuracy_from_logits(x[train:] @ W, y[train:])
    p = orch_dir / f"{name}.json"
    p.write_text(json.dumps({"name": name, "lr": lr, "accuracy": acc}))
    return str(p)

def task_select():
    runs = [json.loads(p.read_text()) for p in orch_dir.glob("train_*.json")]
    winner = max(runs, key=lambda r: r["accuracy"])
    p = orch_dir / "selected.json"
    p.write_text(json.dumps(winner, indent=2))
    return str(p)

def task_register():
    winner = json.loads((orch_dir / "selected.json").read_text())
    manifest = {
        "status": "registered",
        "selected_model": winner,
        "dataset": "prepared.npz",
    }
    p = orch_dir / "registry_entry.json"
    p.write_text(json.dumps(manifest, indent=2))
    return str(p)

tasks = {
    "ingest":      {"deps": [],                         "fn": task_ingest},
    "preprocess":  {"deps": ["ingest"],                 "fn": task_preprocess},
    "train_A":     {"deps": ["preprocess"],             "fn": lambda: task_train("train_A", 0.02)},
    "train_B":     {"deps": ["preprocess"],             "fn": lambda: task_train("train_B", 0.08)},
    "train_C":     {"deps": ["preprocess"],             "fn": lambda: task_train("train_C", 0.20)},
    "select":      {"deps": ["train_A","train_B","train_C"], "fn": task_select},
    "register":    {"deps": ["select"],                 "fn": task_register},
}

pending = set(tasks)
completed = set()
timeline = []

t0 = time.perf_counter()

while pending:
    ready = sorted(
        name for name in pending
        if set(tasks[name]["deps"]).issubset(completed)
    )
    if not ready:
        raise RuntimeError("No runnable tasks. The dependency graph may contain a cycle.")

    wave_start = time.perf_counter() - t0
    with ThreadPoolExecutor(max_workers=len(ready)) as ex:
        future_map = {ex.submit(tasks[name]["fn"]): name for name in ready}
        for fut in as_completed(future_map):
            name = future_map[fut]
            result = fut.result()
            timeline.append({
                "task": name,
                "wave_start_s": wave_start,
                "finished_s": time.perf_counter()-t0,
                "artifact": result,
            })
            completed.add(name)
            pending.remove(name)

print("execution order by completion:")
for item in sorted(timeline, key=lambda x: x["finished_s"]):
    print(
        f"{item['task']:10s} | start wave={item['wave_start_s']:.3f}s | "
        f"finished={item['finished_s']:.3f}s"
    )

print("\nselected model:")
print((orch_dir / "selected.json").read_text())
print("\nregistry entry:")
print((orch_dir / "registry_entry.json").read_text())


### What happened?

The orchestrator did not execute the three training branches until preprocessing completed.

Then:

**train_A, train_B and train_C became ready at the same time**

Only after all three completed could the **select** task run.

Only after selection could the **register** task run.

This dependency logic is a different responsibility from resource allocation by Slurm.


## 14 - [REAL CLUSTER] Map the workflow onto HPC resources

Now inspect the Slurm resources visible from this environment.

Look for:

- partitions;
- GPU generic resources;
- CPUs per node;
- memory;
- your current jobs.

This connects the abstract workflow to real infrastructure.


In [ ]:
if shutil.which("sinfo"):
    print("Visible Slurm partitions and resources:")
    print(sh("sinfo -o '%P  GRES=%G  CPUs=%c  MemMB=%m' | head -20"))
else:
    print("sinfo is not visible in this environment.")

if shutil.which("squeue"):
    user = os.environ.get("USER", "")
    print("\nCurrent jobs for user:", user)
    print(sh(f"squeue -u {user} -o '%.10i %.12P %.24j %.2t %.10M %.6D %R' | head -15"))

if shutil.which("nvidia-smi"):
    print("\nGPU visible inside this Jupyter session:")
    print(sh("nvidia-smi --query-gpu=index,name,memory.total,memory.used --format=csv,noheader | head -8"))


## 14.1 - Two different ways to use a 2-GPU node

A node with two GPUs can support very different workflow shapes.

### A - Experiment throughput

**candidate A -> GPU 0**  
**candidate B -> GPU 1**

Two independent models train at the same time.

### B - One distributed training job

**DDP rank 0 -> GPU 0**  
**DDP rank 1 -> GPU 1**

The two processes cooperate on one model and synchronize gradients.

**Question:** Which pattern improves the throughput of a hyperparameter search, and which pattern tries to reduce the training time of one model?


In [ ]:
array_script = OUT / "slurm_fanout_array.sh"
ddp_slurm_script = OUT / "slurm_ddp_2gpu.sh"

array_script.write_text("""#!/bin/bash
#SBATCH --job-name=m4s4-fanout
#SBATCH --partition=gpu
#SBATCH --array=0-3
#SBATCH --gpus=1
#SBATCH --cpus-per-task=2
#SBATCH --mem=4G
#SBATCH --time=00:05:00

LRS=(0.005 0.02 0.08 0.20)
LR=${LRS[$SLURM_ARRAY_TASK_ID]}

python m4s4_outputs/train_candidate.py \
  --data m4s4_outputs/workflow_dataset.npz \
  --lr $LR \
  --name candidate_$SLURM_ARRAY_TASK_ID \
  --out m4s4_outputs/slurm_candidates
""")

ddp_slurm_script.write_text("""#!/bin/bash
#SBATCH --job-name=m4s4-ddp
#SBATCH --partition=gpu
#SBATCH --gpus=2
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:05:00

module load PyTorch/2.1.2-foss-2023a
torchrun --standalone --nproc_per_node=2 m4s4_outputs/m4s4_ddp_demo.py
""")

print("fan-out job array:")
print(array_script.read_text())
print("\n2-GPU DDP job:")
print(ddp_slurm_script.read_text())


### Optional cluster submission

Do not submit during a lecture unless instructed.

The two files above are intentionally separate:

- **Slurm array:** four independent experiments;
- **2-GPU DDP:** two processes collaborating on one model.

To submit later:

```bash
sbatch m4s4_outputs/slurm_fanout_array.sh
sbatch m4s4_outputs/slurm_ddp_2gpu.sh
```

The exact partition or module name can be adjusted if the cluster configuration changes.


## 15 - [ACTIVITY] Diagnose this distributed ML workflow

A team trains eight model candidates.

1. Every candidate repeats the same expensive preprocessing.
2. A six-hour training job fails after five hours.
3. Model selection waits forever for one failed candidate.
4. Serving latency becomes very high during peak traffic.
5. The team cannot tell which dataset version produced the deployed model.

Choose one or more useful patterns for each problem:

**cache · shard · checkpoint · retry · fan-out/fan-in · replicated serving · batching · metadata · monitoring**

Discuss with a partner before opening the answer.


<details>
<summary>Check one reasonable set of answers</summary>

1. Repeated preprocessing: **cache** the reusable result.
2. Long training failure: **checkpoint + retry/resume**.
3. Selection blocked by failure: **workflow failure policy + retry/timeout + fan-in logic**.
4. Peak serving demand: **replicated serving**, possibly **batching** if the latency objective allows it.
5. Unknown dataset version: **metadata**, content hashes and reproducible artifacts.

There can be more than one valid architecture. The important part is to explain the trade-off.

</details>


# 16 - Exit ticket

A workflow has:

- a dataset split into four pieces;
- four workers training one shared model;
- one worker fails;
- three candidate models are later compared;
- the selected model receives thousands of requests.

Name one relevant pattern for each:

1. **data distribution**
2. **gradient synchronization**
3. **failure recovery**
4. **model selection**
5. **serving scale**

<details>
<summary>Check your answer</summary>

1. Data distribution: **sharding**
2. Gradient synchronization: **collective communication / All-Reduce**
3. Failure recovery: **checkpoint + retry/resume**
4. Model selection: **fan-out/fan-in + evaluation**
5. Serving scale: **replication**, with batching when appropriate

</details>

## Four things to remember

**Training is only one stage.**

**Data movement is part of performance.**

**Parallelism exists within jobs and between jobs.**

**Scale requires orchestration, monitoring and recovery.**
